# Semana 7 · Backpropagation

**Nivel Intermedio** · Semana 7 · Clase expositiva [A] · 2 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/clases/sem07_backpropagation.ipynb)

**Objetivos.** Al terminar la clase podrás:

1. Dibujar una cuenta como un grafo computacional y calcular, en papel, el gradiente de la salida respecto de cada entrada, de atrás hacia adelante.
2. Explicar por qué *backpropagation* calcula todos los gradientes de una red por el precio de unas pocas pasadas, y la derivada numérica no.
3. Escribir en NumPy la pasada hacia atrás (*backward pass*) de un MLP con matrices, y comprobarla con la derivada numérica.
4. Entrenar ese MLP con tu propio gradiente y compararlo con una línea base y con sklearn.

**Prerrequisitos.** Semana 4: derivadas, regla de la cadena, diferencia centrada, *softmax* y entropía cruzada. Semana 6: la propagación hacia adelante, ReLU y la red en matrices, `X @ W.T + b`. Todavía **no** usamos PyTorch (semana 8): hoy los gradientes los calculamos nosotros.

**Hoy se trabaja con lápiz y papel.** Esta es la clase que no se debe delegar: en la semana 8 PyTorch hace todo esto con una sola línea, y solo vas a entender esa línea si hoy la hiciste a mano.

**Agenda** (después de la apertura de 15 minutos, donde se entrega el B3).

| Parte | Tema | Minutos |
|---|---|---|
| 1 | Por qué no basta la derivada numérica | 10 |
| 2 | El grafo computacional | 15 |
| 3 | Tres patrones y una regla para las ramas | 15 |
| 4 | Una neurona, de atrás hacia adelante | 10 |
| 5 | Una red en papel | 20 |
| 6 | La red en matrices | 20 |
| 7 | Entrenar con nuestro propio gradiente | 15 |
| 8 | Cierre y tareas para la semana 8 | 5 |

Ejecuta esta celda primero: carga las librerías y los datos, fija la semilla y define `check`, la función que revisa tus ejercicios de "Tu turno".

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

SEED = 0
rng = np.random.default_rng(SEED)

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data / 16, digits.target, test_size=0.25, stratify=digits.target, random_state=SEED)


def check(name, test):
    """Ejecuta una prueba y muestra ✓, ✗ o "todavía sin resolver", sin detener el notebook."""
    try:
        test()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"… {name}: todavía sin resolver")
    except AssertionError as e:
        print(f"✗ {name}: {e}")
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__}: {e}")


def relu(z):
    return np.maximum(z, 0)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def softmax(Z):
    """Softmax de cada fila de Z (semana 4)."""
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


def cross_entropy(scores, y):
    """Entropía cruzada promedio entre softmax(scores) y las etiquetas enteras y (semana 4)."""
    P = softmax(scores)
    return -np.log(P[np.arange(len(y)), y]).mean()


def numerical_derivative(f, x, h=1e-5):
    """Diferencia centrada (semana 4)."""
    return (f(x + h) - f(x - h)) / (2 * h)


print("dígitos:", X_train.shape, "para entrenar,", X_test.shape, "para probar")

## 1. Por qué no basta la derivada numérica

En la semana 6 la red $64 \to 32 \to 10$ de los dígitos tenía la forma correcta y los números equivocados. Para entrenarla con descenso de gradiente (semana 4) necesitamos, en cada paso, la derivada de la pérdida respecto de **cada uno** de sus parámetros.

Ya tenemos una herramienta que la calcula: la diferencia centrada. Imagina una consola de sonido con 2410 perillas y una canción que suena mal. Para saber hacia dónde girar cada perilla, la giras un poquito a la derecha, escuchas, la giras un poquito a la izquierda, escuchas, y la devuelves a su lugar. Una perilla a la vez. Son **dos pasadas hacia adelante por parámetro**.

La función de abajo hace exactamente eso con un arreglo de parámetros: mueve cada número, mide la pérdida y lo devuelve a su lugar.

In [ ]:
def numerical_grad(f, param, h=1e-5):
    """Gradiente numérico de f() respecto de cada número de param. Mueve cada número y lo devuelve a su lugar."""
    grad = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        old = param[idx]
        param[idx] = old + h
        plus = f()
        param[idx] = old - h
        minus = f()
        param[idx] = old
        grad[idx] = (plus - minus) / (2 * h)
    return grad


def mlp2_loss(X, y, W1, b1, W2, b2):
    """La pérdida de la red de dos capas: ReLU en la oculta, entropía cruzada a la salida."""
    H = relu(X @ W1.T + b1)
    return cross_entropy(H @ W2.T + b2, y)


init_rng = np.random.default_rng(SEED)
W1 = init_rng.normal(size=(32, 64)) * np.sqrt(2 / 64)
b1 = np.zeros(32)
W2 = init_rng.normal(size=(10, 32)) * np.sqrt(2 / 32)
b2 = np.zeros(10)
params = [W1, b1, W2, b2]
n_params = sum(p.size for p in params)


def loss_now():
    return mlp2_loss(X_train, y_train, W1, b1, W2, b2)


start = time.perf_counter()
for _ in range(50):
    loss_now()
t_forward = (time.perf_counter() - start) / 50

start = time.perf_counter()
grads_num = [numerical_grad(loss_now, p) for p in params]
t_numerical = time.perf_counter() - start

print(f"parámetros:                     {n_params}")
print(f"una pasada hacia adelante:      {1000 * t_forward:.2f} ms")
print(f"pasadas para el gradiente:      {2 * n_params}")
print(f"tiempo del gradiente numérico:  {t_numerical:.2f} s, para UN paso de descenso de gradiente")

Un par de segundos por paso parece poco, pero entrenar son cientos o miles de pasos, y el costo crece con el **cuadrado** del tamaño: más parámetros, y cada pasada más lenta. Una red de un millón de parámetros necesitaría dos millones de pasadas por paso.

Hay una forma de obtener **los 2410 números con una pasada hacia adelante y una hacia atrás**, sin importar cuántos parámetros haya. Se llama **backpropagation** (en español, retropropagación) y es la regla de la cadena de la semana 4, aplicada con orden. Al final de la parte 6 vamos a medir cuánto más rápida es, y comprobar que da los mismos números.

## 2. El grafo computacional

Empecemos lejos de las redes, con una compra. Pides 4 camisetas por internet; cada una cuesta 3 dólares, y el envío suma 2 dólares **por camiseta**:

$$\text{subtotal} = \text{precio} + \text{envío} = 3 + 2 = 5 \qquad\qquad \text{total} = \text{subtotal} \times \text{cantidad} = 5 \times 4 = 20$$

Un **grafo computacional** (*computational graph*) dibuja esa cuenta: cada operación es un nodo, y las flechas llevan los números de izquierda a derecha.

```
precio = 3 ──┐
             (+) ── subtotal = 5 ──┐
envío  = 2 ──┘                     (×) ── total = 20
cantidad = 4 ──────────────────────┘
```

La pregunta de backpropagation es: **si cada entrada sube un poquito, ¿cuánto sube el total?** Es decir, $\partial\,\text{total} / \partial\,\text{precio}$, $\partial\,\text{total} / \partial\,\text{envío}$ y $\partial\,\text{total} / \partial\,\text{cantidad}$.

Cada nodo solo conoce su **derivada local** (*local gradient*): cómo cambia su salida cuando cambia una de sus entradas.

| Nodo | Derivada local |
|---|---|
| $\text{total} = \text{subtotal} \times \text{cantidad}$ | $\dfrac{\partial\,\text{total}}{\partial\,\text{subtotal}} = \text{cantidad} = 4 \qquad \dfrac{\partial\,\text{total}}{\partial\,\text{cantidad}} = \text{subtotal} = 5$ |
| $\text{subtotal} = \text{precio} + \text{envío}$ | $\dfrac{\partial\,\text{subtotal}}{\partial\,\text{precio}} = 1 \qquad \dfrac{\partial\,\text{subtotal}}{\partial\,\text{envío}} = 1$ |

Ahora recorremos el grafo **de derecha a izquierda**. Empezamos en el total con un 1 (el total cambia exactamente lo que cambia el total) y en cada nodo aplicamos una sola regla:

$$\boxed{\text{gradiente de una entrada} = \text{gradiente que llega} \times \text{derivada local}}$$

1. Total: $1$.
2. Subtotal: $1 \times 4 = 4$. Cantidad: $1 \times 5 = 5$.
3. Precio: $4 \times 1 = 4$. Envío: $4 \times 1 = 4$.

Se lee así: si el precio sube 1 centavo, el total sube 4 centavos (es lógico: compras 4 camisetas). Si compras una camiseta más, el total sube 5 dólares. El "gradiente que llega" (*upstream gradient*) es la regla de la cadena de la semana 4, pero ya calculada: no hay que volver a multiplicar todos los eslabones desde el final, solo el último.

In [ ]:
price, shipping, quantity = 3.0, 2.0, 4.0

# hacia adelante
subtotal = price + shipping
total = subtotal * quantity

# hacia atrás: gradiente que llega × derivada local
d_total = 1.0
d_subtotal = d_total * quantity
d_quantity = d_total * subtotal
d_price = d_subtotal * 1.0
d_shipping = d_subtotal * 1.0
print(f"total = {total}")
print(f"backprop:  d_price = {d_price}, d_shipping = {d_shipping}, d_quantity = {d_quantity}")

compra = lambda p, s, q: (p + s) * q
print("numérica:  d_price = {:.4f}, d_shipping = {:.4f}, d_quantity = {:.4f}".format(
    numerical_derivative(lambda p: compra(p, shipping, quantity), price),
    numerical_derivative(lambda s: compra(price, s, quantity), shipping),
    numerical_derivative(lambda q: compra(price, shipping, q), quantity)))

## 3. Tres patrones y una regla para las ramas

En una red casi todos los nodos son de tres tipos, y cada uno trata al gradiente que llega de una forma fácil de recordar:

| Nodo | Hacia adelante | Hacia atrás | Imagen |
|---|---|---|---|
| Suma | $a + b$ | **copia** el gradiente a las dos entradas | un repartidor: a cada uno le toca lo mismo |
| Multiplicación | $a \cdot b$ | a cada entrada le da el gradiente **por la otra** entrada | un intercambio: $a$ recibe $b$, $b$ recibe $a$ |
| Máximo, ReLU | $\max(a, b)$ | el gradiente va **entero al ganador**, 0 al otro | una válvula: solo pasa por donde pasó el valor |

ReLU es un máximo contra 0: $\text{ReLU}(z) = \max(z, 0)$. Si $z > 0$, el gradiente pasa entero; si no, no pasa nada.

**La regla de las ramas.** Si un mismo valor se usa en dos lugares, recibe gradiente por los dos caminos, y **los gradientes se suman**. Es como un empleado que trabaja en dos proyectos: su aporte total es la suma de lo que aportó en cada uno. Ejemplo: $f = x \cdot x$. La multiplicación le da a la primera $x$ el valor de la segunda, y viceversa: $x + x = 2x$, que es justo la derivada de $x^2$.

In [ ]:
x = 3.0
f = x * x
d_first, d_second = x, x          # la multiplicación intercambia
d_x = d_first + d_second          # x se usó dos veces: los gradientes se suman
print(f"backprop: {d_x}   numérica: {numerical_derivative(lambda t: t * t, x):.4f}")

### Tu turno 1: dos grafos en papel

Con $a = 1$, $b = 2$, $c = 3$. **Dibuja cada grafo, escribe los valores hacia adelante y después los gradientes hacia atrás.** Sin ejecutar nada.

1. $f = (a + b) \cdot (b + c)$. Ojo: $b$ se usa dos veces. Escribe `df_da`, `df_db` y `df_dc`.
2. $g = \max(a, c) \cdot b$. Escribe `dg_da`, `dg_db` y `dg_dc`.

In [ ]:
df_da, df_db, df_dc = None, None, None  # TODO
dg_da, dg_db, dg_dc = None, None, None  # TODO

In [ ]:
def test_branches():
    if None in (df_da, df_db, df_dc):
        raise NotImplementedError
    f = lambda a, b, c: (a + b) * (b + c)
    assert np.isclose(df_da, numerical_derivative(lambda t: f(t, 2, 3), 1.0)), "revisa df_da: la suma copia el gradiente"
    assert np.isclose(df_dc, numerical_derivative(lambda t: f(1, 2, t), 3.0)), "revisa df_dc"
    assert np.isclose(df_db, numerical_derivative(lambda t: f(1, t, 3), 2.0)), "revisa df_db: b llega por dos caminos"


def test_max():
    if None in (dg_da, dg_db, dg_dc):
        raise NotImplementedError
    g = lambda a, b, c: max(a, c) * b
    assert np.isclose(dg_da, numerical_derivative(lambda t: g(t, 2, 3), 1.0)), "revisa dg_da: ¿quién ganó el máximo?"
    assert np.isclose(dg_dc, numerical_derivative(lambda t: g(1, 2, t), 3.0)), "revisa dg_dc"
    assert np.isclose(dg_db, numerical_derivative(lambda t: g(1, t, 3), 2.0)), "revisa dg_db: la multiplicación intercambia"


check("f = (a + b)(b + c)", test_branches)
check("g = max(a, c) · b", test_max)

## 4. Una neurona, de atrás hacia adelante

Volvamos a la neurona de la semana 4: $x = 2$, $w = 0.5$, $b = -1$, la respuesta correcta es $y = 1$, y la pérdida es $L = (a - y)^2$.

| Hacia adelante | Valor | Hacia atrás | Valor |
|---|---|---|---|
| $z = w x + b$ | $0$ | $\partial L / \partial z = -1 \times \sigma'(0) = -1 \times 0.25$ | $-0.25$ |
| $a = \sigma(z)$ | $0.5$ | $\partial L / \partial a = 2(a - y)$ | $-1$ |
| $L = (a - y)^2$ | $0.25$ | $\partial L / \partial L$ | $1$ |

La columna de la derecha se lee **de abajo hacia arriba**. Desde $z$, los tres parámetros salen con los patrones de la parte 3: $wx$ es una multiplicación y $+\,b$ una suma.

$$\frac{\partial L}{\partial w} = -0.25 \times x = -0.5 \qquad \frac{\partial L}{\partial x} = -0.25 \times w = -0.125 \qquad \frac{\partial L}{\partial b} = -0.25$$

El $-0.5$ es el número de la semana 4. Lo nuevo es que **el $-0.25$ se calculó una sola vez y sirvió para los tres**. Eso es lo que hace barato a backpropagation: cada gradiente se calcula una vez y se reutiliza para todo lo que está antes de él. Con la derivada numérica, en cambio, cada parámetro repite la cuenta entera.

In [ ]:
x, w, b, y = 2.0, 0.5, -1.0, 1.0

# hacia adelante: guardamos lo que la pasada hacia atrás va a necesitar
z = w * x + b
a = sigmoid(z)
L = (a - y) ** 2

# hacia atrás
dL_da = 2 * (a - y)
dL_dz = dL_da * a * (1 - a)      # σ'(z) = σ(z)(1 − σ(z)), y σ(z) ya lo tenemos: es a
dL_dw = dL_dz * x
dL_dx = dL_dz * w
dL_db = dL_dz

neuron = lambda w_, x_, b_: (sigmoid(w_ * x_ + b_) - y) ** 2
print(f"dL/dw = {dL_dw:.4f}   numérica: {numerical_derivative(lambda t: neuron(t, x, b), w):.4f}")
print(f"dL/dx = {dL_dx:.4f}   numérica: {numerical_derivative(lambda t: neuron(w, t, b), x):.4f}")
print(f"dL/db = {dL_db:.4f}   numérica: {numerical_derivative(lambda t: neuron(w, x, t), b):.4f}")

Fíjate en un detalle: para calcular $\sigma'(z)$ no hizo falta volver a calcular $\sigma$; usamos la $a$ que guardamos al ir hacia adelante. Por eso la pasada hacia adelante **guarda** sus valores intermedios: la pasada hacia atrás los va a necesitar.

### La pieza final: sigmoide con entropía cruzada

Para clasificar no usamos el error cuadrático sino la entropía cruzada binaria (semanas 4 y 5), con $p = \sigma(z)$:

$$L = -\left[\, y \ln p + (1 - y) \ln (1 - p) \,\right]$$

Dos eslabones. El de la pérdida, $\dfrac{\partial L}{\partial p} = -\dfrac{y}{p} + \dfrac{1 - y}{1 - p} = \dfrac{p - y}{p\,(1 - p)}$, y el de la sigmoide, $\dfrac{\partial p}{\partial z} = p\,(1 - p)$. Al multiplicarlos, el denominador se cancela:

$$\boxed{\frac{\partial L}{\partial z} = p - y}$$

Es el resultado más usado de esta clase: el gradiente en la salida es **cuánto te pasaste**. Si dijiste 0.82 y era 1, es $-0.18$. Es el mismo $p - y$ que usa la regresión logística.

## 5. Una red en papel

Ahora la red de la semana 6: 2 entradas, 2 neuronas ocultas con ReLU y una salida con sigmoide. Con $x = (2, -1)$ ya hicimos la ida:

$$W_1 = \begin{pmatrix} 1 & 2 \\ -1 & 1 \end{pmatrix} \quad b_1 = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \qquad\qquad W_2 = \begin{pmatrix} 2 & -1 \end{pmatrix} \quad b_2 = -0.5$$

| Hacia adelante | Valor |
|---|---|
| $a = W_1 x + b_1$ | $(1, -3)$ |
| $h = \text{ReLU}(a)$ | $(1, 0)$ |
| $z = W_2 h + b_2$ | $1.5$ |
| $p = \sigma(z)$ | $0.818$ |

La respuesta correcta es $y = 1$. Ahora la vuelta, con entropía cruzada. Cada fila usa solo la fila anterior y algo que guardamos al ir:

| Hacia atrás | Cuenta | Valor |
|---|---|---|
| $\partial L / \partial z$ | $p - y = 0.818 - 1$ | $-0.182$ |
| $\partial L / \partial W_2$ | multiplicación: $\partial L / \partial z \times h = -0.182 \times (1, 0)$ | $(-0.182,\ 0)$ |
| $\partial L / \partial b_2$ | suma: se copia | $-0.182$ |
| $\partial L / \partial h$ | multiplicación: $\partial L / \partial z \times W_2 = -0.182 \times (2, -1)$ | $(-0.365,\ 0.182)$ |
| $\partial L / \partial a$ | ReLU: $a_1 = 1 > 0$ pasa, $a_2 = -3$ no pasa | $(-0.365,\ 0)$ |
| $\partial L / \partial W_1$ | fila $i$, columna $j$: $\partial L / \partial a_i \times x_j$ | $\begin{pmatrix} -0.730 & 0.365 \\ 0 & 0 \end{pmatrix}$ |
| $\partial L / \partial b_1$ | suma: se copia | $(-0.365,\ 0)$ |

Mira la segunda fila de $\partial L / \partial W_1$: **ceros**. La neurona $h_2$ estaba apagada para esta entrada, así que no influyó en la salida y no recibe culpa. Sus pesos no se mueven con este ejemplo.

¿Y el signo? Todos los gradientes de la neurona $h_1$ son negativos, entonces el descenso de gradiente, que resta, va a **subir** sus pesos: $h_1$ empujaba hacia la respuesta correcta, y la red le da más voz. Comprobémoslo con la derivada numérica:

In [ ]:
x = np.array([2.0, -1.0])
y = 1.0
W1_p = np.array([[1.0, 2.0], [-1.0, 1.0]])
b1_p = np.array([1.0, 0.0])
W2_p = np.array([[2.0, -1.0]])
b2_p = np.array([-0.5])


def paper_loss(x, y, W1, b1, W2, b2):
    p = sigmoid(W2 @ relu(W1 @ x + b1) + b2)[0]
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


# hacia adelante, guardando a y h
a = W1_p @ x + b1_p
h = relu(a)
p = sigmoid(W2_p @ h + b2_p)      # forma (1,): una sola salida

# hacia atrás, fila por fila de la tabla
dz = p - y
dW2 = np.outer(dz, h)             # cada peso: el gradiente de su salida por su entrada
db2 = dz
dh = dz @ W2_p
da = dh * (a > 0)
dW1 = np.outer(da, x)
db1 = da

loss_p = lambda: paper_loss(x, y, W1_p, b1_p, W2_p, b2_p)
for name, ours, param in [("W1", dW1, W1_p), ("b1", db1, b1_p), ("W2", dW2, W2_p), ("b2", db2, b2_p)]:
    print(f"{name}  backprop: {(np.round(ours, 3) + 0.0).tolist()}")
    print(f"    numérica: {(np.round(numerical_grad(loss_p, param), 3) + 0.0).tolist()}")

### Tu turno 2: la vuelta de tu red de papel

En el Tu turno 3 de la semana 6 hiciste la ida de esta red, con $x = (1, 2)$:

$$h_1 = \text{ReLU}(1 \cdot x_1 - 2 \cdot x_2 + 1) \qquad h_2 = \text{ReLU}(-1 \cdot x_1 + 3 \cdot x_2 + 0) \qquad z = 1 \cdot h_1 - 0.5 \cdot h_2 + 2$$

Te había dado $h = (0, 5)$ y $z = -0.5$, y la red decía clase 0. Pero la respuesta correcta es $y = 1$. Te damos $p = \sigma(-0.5) = 0.378$. **Con lápiz, sigue la tabla de arriba fila por fila** y escribe:

- `paper_dz`: $\partial L / \partial z$.
- `paper_dW2`: los dos números de $\partial L / \partial W_2$.
- `paper_dh`: los dos números de $\partial L / \partial h$.
- `paper_dW1`: la matriz de $2 \times 2$ de $\partial L / \partial W_1$, como lista de dos filas.

Después responde en una línea: ¿qué fila de $\partial L / \partial W_1$ es cero esta vez, y por qué?

In [ ]:
paper_dz = None   # TODO
paper_dW2 = None  # TODO, por ejemplo [0.1, -0.2]
paper_dh = None   # TODO
paper_dW1 = None  # TODO, por ejemplo [[0.1, 0.2], [0.3, 0.4]]

In [ ]:
def test_paper_backward():
    if None in (paper_dz, paper_dW2, paper_dh, paper_dW1):
        raise NotImplementedError
    x = np.array([1.0, 2.0])
    W1 = np.array([[1.0, -2.0], [-1.0, 3.0]])
    b1 = np.array([1.0, 0.0])
    W2 = np.array([[1.0, -0.5]])
    b2 = np.array([2.0])
    a = W1 @ x + b1
    h = relu(a)
    dz = sigmoid(W2 @ h + b2)[0] - 1
    assert np.isclose(paper_dz, dz, atol=0.01), "revisa dz: es p − y"
    assert np.allclose(paper_dW2, dz * h, atol=0.01), "revisa dW2: dz por lo que entró a la capa, h"
    assert np.allclose(paper_dh, dz * W2[0], atol=0.01), "revisa dh: dz por los pesos de W2"
    assert np.allclose(paper_dW1, np.outer(dz * W2[0] * (a > 0), x), atol=0.01), \
        "revisa dW1: primero pasa dh por ReLU, después multiplica por x"


check("la vuelta en papel", test_paper_backward)

## 6. La red en matrices

Con 2 neuronas, la tabla cabe en una hoja. Con 32 neuronas y 1347 ejemplos, no. Igual que en la semana 6 la ida se escribió como `X @ W.T + b`, la vuelta también se escribe con matrices, y son **tres fórmulas para cualquier capa lineal**.

Una capa recibe $X$ con forma `(n, entradas)` y calcula $Z = X W^\top + b$, con $W$ de forma `(salidas, entradas)`. Desde arriba le llega $G = \partial L / \partial Z$, con la misma forma que $Z$: `(n, salidas)`. Entonces:

$$\frac{\partial L}{\partial W} = G^\top X \qquad\qquad \frac{\partial L}{\partial b} = \sum_{\text{filas}} G \qquad\qquad \frac{\partial L}{\partial X} = G\, W$$

Son las mismas reglas de la tabla en papel: el peso $W_{ij}$ une la entrada $j$ con la salida $i$, así que recibe "gradiente de la salida $i$ por la entrada $j$", sumado sobre todos los ejemplos. Eso es $G^\top X$.

**El truco de las formas.** No hace falta memorizar las fórmulas: el gradiente de algo tiene la misma forma que ese algo, y **solo hay una manera** de combinar $G$ con lo que tienes para obtener esa forma.

| Quieres | Forma | Tienes | Única combinación |
|---|---|---|---|
| $\partial L / \partial W$ | `(salidas, entradas)` | `G` `(n, salidas)`, `X` `(n, entradas)` | `G.T @ X` |
| $\partial L / \partial b$ | `(salidas,)` | `G` `(n, salidas)` | `G.sum(axis=0)` |
| $\partial L / \partial X$ | `(n, entradas)` | `G` `(n, salidas)`, `W` `(salidas, entradas)` | `G @ W` |

Faltan dos piezas que no tienen pesos:

- **ReLU**, la válvula, ahora con una matriz: `G * (Z > 0)`. El gradiente pasa solo donde $Z$ era positivo.
- **Softmax con entropía cruzada**, la promesa de la semana 4. Para un ejemplo de la clase $c$, con $p_k = e^{z_k} / \sum_j e^{z_j}$:

$$L = -\ln p_c = -z_c + \ln \sum_j e^{z_j} \qquad\Longrightarrow\qquad \frac{\partial L}{\partial z_k} = -[k = c] + \frac{e^{z_k}}{\sum_j e^{z_j}} = p_k - [k = c]$$

donde $[k = c]$ vale 1 si $k$ es la clase correcta y 0 si no. Es $p - y$ otra vez, ahora con una fila por ejemplo: $P - Y$, con $Y$ en *one-hot*. Como la pérdida es el **promedio** sobre los $n$ ejemplos, cada fila se divide entre $n$: $\partial L / \partial Z = (P - Y)/n$.

Con eso, la red entera de los dígitos, ida y vuelta:

In [ ]:
def forward(X, W1, b1, W2, b2):
    """La ida. Devuelve los puntajes y lo que la vuelta va a necesitar."""
    Z1 = X @ W1.T + b1
    H = relu(Z1)
    scores = H @ W2.T + b2
    return scores, (X, Z1, H)


def backward(scores, cache, y, W2):
    """La vuelta. Devuelve los gradientes de la entropía cruzada promedio, en el orden W1, b1, W2, b2."""
    X, Z1, H = cache
    n = len(y)
    G2 = softmax(scores)
    G2[np.arange(n), y] -= 1
    G2 /= n                       # (P − Y) / n
    dW2 = G2.T @ H
    db2 = G2.sum(axis=0)
    dH = G2 @ W2
    G1 = dH * (Z1 > 0)            # ReLU: pasa solo donde estaba abierta
    dW1 = G1.T @ X
    db1 = G1.sum(axis=0)
    return dW1, db1, dW2, db2


start = time.perf_counter()
for _ in range(50):
    scores, cache = forward(X_train, W1, b1, W2, b2)
    grads_bp = backward(scores, cache, y_train, W2)
t_backprop = (time.perf_counter() - start) / 50

print(f"gradiente numérico (parte 1):   {t_numerical:.2f} s")
print(f"backpropagation (ida y vuelta): {1000 * t_backprop:.2f} ms")
print(f"backpropagation es unas {t_numerical / t_backprop:,.0f} veces más rápida")

Más rápida, sí, pero ¿da lo mismo? Para comparar dos arreglos de gradientes se usa el **error relativo**:

$$\text{error relativo} = \frac{\lVert a - b \rVert}{\lVert a \rVert + \lVert b \rVert}$$

No depende de la escala de los números. Una regla práctica: menos de $10^{-7}$ es que coinciden; más de $10^{-4}$ es que hay un error en tu vuelta. Comprobar así un gradiente (*gradient checking*) es la forma de saber si una vuelta escrita a mano está bien. **Los profesionales lo hacen siempre que escriben una.**

In [ ]:
def rel_error(a, b):
    return np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12)


for name, g_bp, g_num in zip(["W1", "b1", "W2", "b2"], grads_bp, grads_num):
    print(f"{name}: forma {str(g_bp.shape):<9} error relativo {rel_error(g_bp, g_num):.1e}")

Los mismos 2410 números, cientos o miles de veces más rápido (depende de la máquina). Y la ventaja crece con la red: el gradiente numérico cuesta dos pasadas **por parámetro**; backpropagation, unas pocas pasadas **en total**.

### Tu turno 3: la vuelta para cualquier profundidad

En el Tu turno 4 de la semana 6 escribiste `mlp_forward(X, weights, biases)` para una lista de capas de cualquier largo, con ReLU después de cada capa menos la última. Ahora la vuelta.

Completa `mlp_backward(X, y, weights, biases)`. Devuelve dos listas, `dWs` y `dbs`, con el gradiente de la entropía cruzada promedio respecto de cada `W` y cada `b`, en el mismo orden que `weights` y `biases`.

1. **Ida.** Recorre las capas como en `mlp_forward`, pero **guarda** en listas la entrada de cada capa y su `Z`. La vuelta los necesita.
2. **Salida.** `G = (P - Y) / n`, igual que en `backward`.
3. **Vuelta.** Recorre las capas de la última a la primera (`reversed(range(len(weights)))`). En cada una calcula `dW` y `db` con las fórmulas, y el `G` de la capa anterior: primero `G @ W`, después la válvula ReLU de la capa anterior.

Escríbelo sin mirar `backward`; si te trabas, vuelve a la tabla de las formas.

In [ ]:
def mlp_backward(X, y, weights, biases):
    # TODO
    raise NotImplementedError

In [ ]:
def deep_loss(X, y, weights, biases):
    A = X
    for i, (W, b) in enumerate(zip(weights, biases)):
        A = A @ W.T + b
        if i < len(weights) - 1:
            A = relu(A)
    return cross_entropy(A, y)


def test_two_layers():
    dWs, dbs = mlp_backward(X_train, y_train, [W1, W2], [b1, b2])
    assert len(dWs) == 2 and len(dbs) == 2, "debe devolver un gradiente por capa"
    for got, want, name in zip(dWs + dbs, [grads_bp[0], grads_bp[2], grads_bp[1], grads_bp[3]], ["W1", "W2", "b1", "b2"]):
        assert got.shape == want.shape, f"d{name} debería tener forma {want.shape}, tiene {got.shape}"
        assert rel_error(got, want) < 1e-7, f"d{name} no coincide con backward"


def test_three_layers():
    r = np.random.default_rng(1)
    sizes = [5, 4, 6, 3]
    Ws = [r.normal(size=(o, i)) for i, o in zip(sizes[:-1], sizes[1:])]
    bs = [r.normal(size=o) for o in sizes[1:]]
    Xs, ys = r.normal(size=(8, 5)), r.integers(0, 3, size=8)
    dWs, dbs = mlp_backward(Xs, ys, Ws, bs)
    assert len(dWs) == 3, "con tres capas debe devolver tres gradientes"
    for k, (W, b) in enumerate(zip(Ws, bs)):
        num_W = numerical_grad(lambda: deep_loss(Xs, ys, Ws, bs), W)
        num_b = numerical_grad(lambda: deep_loss(Xs, ys, Ws, bs), b)
        assert rel_error(dWs[k], num_W) < 1e-6, f"el gradiente de la capa {k} (W) no coincide con el numérico"
        assert rel_error(dbs[k], num_b) < 1e-6, f"el gradiente de la capa {k} (b) no coincide con el numérico"


check("mlp_backward con dos capas", test_two_layers)
check("mlp_backward con tres capas", test_three_layers)

## 7. Entrenar con nuestro propio gradiente

Ya tenemos todo: la ida, la vuelta y el descenso de gradiente de la semana 4, $\theta \leftarrow \theta - \eta\, \nabla L$. Un paso de entrenamiento es esas tres cosas, en ese orden.

Los pesos empiezan al azar, con una escala que depende de cuántas entradas tiene la capa. Por qué esa escala y no otra es tema de la semana 9; por ahora basta con que no empiecen todos iguales (si empezaran iguales, todas las neuronas recibirían el mismo gradiente y harían siempre lo mismo).

In [ ]:
def init_params(sizes, seed=SEED):
    """Pesos al azar y sesgos en cero para una red de dos capas con tamaños (entradas, ocultas, clases)."""
    r = np.random.default_rng(seed)
    d, hidden, k = sizes
    return [r.normal(size=(hidden, d)) * np.sqrt(2 / d), np.zeros(hidden),
            r.normal(size=(k, hidden)) * np.sqrt(2 / hidden), np.zeros(k)]


def train(X, y, sizes, lr, steps):
    params = init_params(sizes)
    losses = []
    for _ in range(steps):
        scores, cache = forward(X, *params)                  # 1. ida
        losses.append(cross_entropy(scores, y))
        grads = backward(scores, cache, y, params[2])         # 2. vuelta
        for p, g in zip(params, grads):
            p -= lr * g                                       # 3. un paso cuesta abajo
    return params, losses


def predict(params, X):
    return forward(X, *params)[0].argmax(axis=1)

### Primero XOR

En la semana 6, XOR con ruido necesitó una capa oculta, y `MLPClassifier` la entrenó por dentro. Ahora la entrenamos nosotros, con 8 neuronas ocultas:

In [ ]:
data_rng = np.random.default_rng(SEED)
corners = data_rng.integers(0, 2, size=(400, 2))
X_xor = corners + data_rng.normal(0, 0.2, size=(400, 2))
y_xor = corners[:, 0] ^ corners[:, 1]
Xx_train, Xx_test, yx_train, yx_test = train_test_split(X_xor, y_xor, test_size=0.25, stratify=y_xor, random_state=SEED)

xor_params, xor_losses = train(Xx_train, yx_train, (2, 8, 2), lr=0.5, steps=2000)
xor_base = DummyClassifier(strategy="most_frequent").fit(Xx_train, yx_train).score(Xx_test, yx_test)
print(f"línea base:  accuracy de prueba {xor_base:.3f}")
print(f"nuestra red: accuracy de prueba {np.mean(predict(xor_params, Xx_test) == yx_test):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(xor_losses, color="tab:blue")
axes[0].axhline(np.log(2), color="gray", linestyle="--", label="adivinar: ln 2")
axes[0].set(xlabel="paso", ylabel="entropía cruzada", title="La pérdida, paso a paso")
axes[0].legend(fontsize=8)

gx, gy = np.meshgrid(np.linspace(-0.8, 1.8, 300), np.linspace(-0.8, 1.8, 300))
region = predict(xor_params, np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
axes[1].contourf(gx, gy, region, levels=[-0.5, 0.5, 1.5], colors=["tab:blue", "tab:orange"], alpha=0.2)
for c, color in [(0, "tab:blue"), (1, "tab:orange")]:
    axes[1].scatter(*Xx_test[yx_test == c].T, s=14, color=color, edgecolor="white", label=f"clase {c}")
axes[1].set(xlabel="$x_1$", ylabel="$x_2$", title="Las regiones que aprendió")
axes[1].legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()

La pérdida arranca cerca de $\ln 2 \approx 0.69$, la de adivinar con dos clases (semana 4); un poco por encima, porque los pesos al azar a veces se equivocan con confianza. Después cae en pocos cientos de pasos. Las regiones son las cuatro esquinas de XOR, y nadie le dijo a la red qué tenían que hacer sus neuronas ocultas: lo encontró el gradiente.

### Después, los dígitos

La misma red de la parte 1, $64 \to 32 \to 10$, entrenada con nuestra vuelta, contra la línea base, la regresión logística y `MLPClassifier` con la misma forma:

In [ ]:
start = time.perf_counter()
digit_params, digit_losses = train(X_train, y_train, (64, 32, 10), lr=0.5, steps=500)
t_train = time.perf_counter() - start

results = {
    "línea base (la clase más común)": DummyClassifier(strategy="most_frequent").fit(X_train, y_train).score(X_test, y_test),
    "regresión logística": LogisticRegression(max_iter=2000).fit(X_train, y_train).score(X_test, y_test),
    "nuestra red 64 → 32 → 10": np.mean(predict(digit_params, X_test) == y_test),
    "MLPClassifier 64 → 32 → 10": MLPClassifier((32,), max_iter=1000, random_state=SEED).fit(X_train, y_train).score(X_test, y_test),
}
for name, acc in results.items():
    print(f"{name:<34} accuracy de prueba {acc:.3f}")
print(f"\n500 pasos en {t_train:.1f} s. Con el gradiente numérico habrían sido unos {500 * t_numerical / 60:.0f} minutos.")

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(digit_losses, color="tab:blue")
ax.axhline(np.log(10), color="gray", linestyle="--", label="adivinar: ln 10")
ax.set(xlabel="paso", ylabel="entropía cruzada", title="Dígitos: la pérdida, paso a paso", ylim=(0, 2.6))
ax.legend(fontsize=8)
plt.show()

Nuestra red, con unas 30 líneas de NumPy escritas hoy, juega en la misma liga que la de sklearn. `MLPClassifier` usa por dentro esta misma vuelta; la diferencia está en cómo da los pasos (Adam, mini-lotes), que es el tema de la semana 10.

### Tu turno 4: cazar un error

Escribir la vuelta a mano tiene un riesgo: un error pequeño no hace que el código falle. La red entrena igual, solo que peor, y nadie se entera. Por eso existe la comprobación del gradiente.

`buggy_backward` tiene **un** error. Tu trabajo:

1. Calcula sus gradientes sobre `X_small, y_small` (20 dígitos) y compáralos, uno por uno, con `numerical_grad` usando `rel_error`. Usa la función `small_loss` para el gradiente numérico.
2. Escribe en `wrong_params` los nombres de los parámetros cuyo gradiente está mal, por ejemplo `["W2"]`.
3. Encuentra la línea con el error y escribe `fixed_backward`, una copia de `buggy_backward` con esa línea corregida.

Pregunta para pensar: el error está en una sola línea, pero ¿afecta a todos los parámetros? ¿Por qué los que salen bien salen bien?

In [ ]:
def buggy_backward(scores, cache, y, W2):
    X, Z1, H = cache
    n = len(y)
    G2 = softmax(scores)
    G2[np.arange(n), y] -= 1
    G2 /= n
    dW2 = G2.T @ H
    db2 = G2.sum(axis=0)
    dH = G2 @ W2
    G1 = dH
    dW1 = G1.T @ X
    db1 = G1.sum(axis=0)
    return dW1, db1, dW2, db2


X_small, y_small = X_train[:20], y_train[:20]
bug_params = init_params((64, 32, 10), seed=3)


def small_loss():
    return mlp2_loss(X_small, y_small, *bug_params)


scores_small, cache_small = forward(X_small, *bug_params)
grads_buggy = buggy_backward(scores_small, cache_small, y_small, bug_params[2])

In [ ]:
# TODO: compara cada gradiente de grads_buggy con numerical_grad(small_loss, ...)
wrong_params = None  # TODO


def fixed_backward(scores, cache, y, W2):
    # TODO
    raise NotImplementedError

In [ ]:
def test_wrong_params():
    if wrong_params is None:
        raise NotImplementedError
    assert sorted(wrong_params) == ["W1", "b1"], \
        f"dijiste {sorted(wrong_params)}; mira de nuevo el error relativo de cada uno"


def test_fixed():
    grads = fixed_backward(scores_small, cache_small, y_small, bug_params[2])
    for name, g, p in zip(["W1", "b1", "W2", "b2"], grads, bug_params):
        assert rel_error(g, numerical_grad(small_loss, p)) < 1e-6, f"d{name} todavía no coincide con el numérico"


check("qué parámetros están mal", test_wrong_params)
check("fixed_backward", test_fixed)

## 8. Cierre

**Hoy vimos:**

- El gradiente numérico cuesta dos pasadas por parámetro: impagable para una red real. Backpropagation da todos los gradientes con una ida y una vuelta.
- Un grafo computacional dibuja la cuenta. De atrás hacia adelante, una sola regla: **gradiente de una entrada = gradiente que llega × derivada local**.
- Tres patrones: la suma copia, la multiplicación intercambia, el máximo (y ReLU) manda todo al ganador. Si un valor se usa dos veces, sus gradientes se suman.
- La ida guarda sus valores intermedios, porque la vuelta los necesita. Cada gradiente se calcula una vez y se reutiliza: por eso es barato.
- Con sigmoide o *softmax* y entropía cruzada, el gradiente en la salida es $p - y$: cuánto te pasaste.
- Para una capa $Z = XW^\top + b$: $\partial L/\partial W = G^\top X$, $\partial L/\partial b = \sum G$, $\partial L/\partial X = G W$. Las formas te dicen cuál es cuál.
- Una vuelta escrita a mano se comprueba siempre con el gradiente numérico y el error relativo.

**Un poco de historia.** La idea de recorrer la regla de la cadena hacia atrás se publicó en 1970 (Seppo Linnainmaa), pero se popularizó para redes neuronales en 1986, con un artículo de David Rumelhart, Geoffrey Hinton y Ronald Williams. Era la pieza que faltaba desde *Perceptrons* (1969): cómo entrenar las capas ocultas.

**Para la semana 8:**

1. **Entrega al inicio de la clase**: **P1: Motor de diferenciación automática desde cero** ([enunciado](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/tareas/P1_autograd_desde_cero.ipynb)). Hoy escribimos la vuelta a mano para una red fija. En P1 vas a automatizarla: cada operación recuerda sus entradas y su derivada local, y una sola llamada recorre el grafo hacia atrás, al estilo de *micrograd*. Con ese motor vas a entrenar un MLP sobre "two moons". Se guarda en tu repositorio de entregas como `P1/P1_autograd_desde_cero.ipynb`.
2. **Cuota semanal** en [la plataforma](https://ioai.artix.tech/).
3. **Para ver** (en inglés, opcionales pero muy recomendados): [What is backpropagation really doing?](https://www.youtube.com/watch?v=Ilg3gGewQ5U) y [Backpropagation calculus](https://www.youtube.com/watch?v=tIeHLnjs5U8) de 3Blue1Brown. Para P1, [The spelled-out intro to neural networks and backpropagation: building micrograd](https://www.youtube.com/watch?v=VMj-3S1tku0) de Andrej Karpathy.

La semana 8 llega PyTorch. Su `loss.backward()` hace exactamente lo de hoy, para cualquier grafo: después de esta clase, ya sabes qué hay dentro de esa línea.